<a href="https://colab.research.google.com/github/awinarko-hue/Latihan/blob/main/DataSains/Prediksi_Durasi_LHU_SLA_ISO17025_Rd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prediksi Durasi Pengujian LHU SLA ISO/IEC 17025

**Dataset**: `Selesai_Pengujian_LHU_ALL_2023.csv` — data Laporan Hasil Uji (LHU) pengujian alat telekomunikasi tahun 2023-2025.

**Tujuan**: Membangun model machine learning untuk memprediksi `Jumlah Hari Uji` berdasarkan karakteristik perangkat dan fitur teknis yang diuji, sebagai dukungan analisis Service Level Agreement (SLA) sesuai ISO/IEC 17025.

**Struktur notebook**:
1. Setup & Load Data
2. Eksplorasi Data Awal (EDA)
3. Feature Engineering — Kategorisasi Perangkat & Label Teknologi RF
4. Pemodelan: Komparasi Machine Learning vs Deep Learning
5. Hyperparameter Tuning (Redam Overfitting)
6. Eksperimen Augmentasi Data Sintetis (SMOTE untuk Regresi)
7. Ringkasan & Kesimpulan

---


## 1. Setup & Load Data

Jalankan cell di bawah ini untuk install dependency yang belum tersedia secara default di Google Colab.

In [ ]:
# Install library
!pip install xgboost -q


In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


### Upload file dataset

Jalankan cell di bawah untuk upload file CSV `Selesai_Pengujian_LHU_ALL_2023.csv` dari komputer Anda ke Colab.

In [ ]:
from google.colab import files
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
file1 = "/content/drive/MyDrive/Kuliah/Project5/Selesai Pengujian LHU ALL 2023.csv"
file2 = "/content/drive/MyDrive/Kuliah/Project5/Selesai Pengujian LHU ALL 2024.csv"
file3 = "//content/drive/MyDrive/Kuliah/Project5/Selesai Pengujian LHU ALL 2025.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)
df3 = pd.read_csv(file3)



df = pd.concat([df1, df2, df3], ignore_index=True)
print("Shape:", df.shape)
df.head()


Shape: (3519, 11)


,No,Nomor Permohonan,Nama Perangkat,Merk / Type,Tanggal Permohonan,Nomor LHU,Tanggal LHU,Jumlah Hari Uji,Fitur,Status,Simpel
0,1.0,0437,EXTERNAL RADIO,Merk : STONEXType : SR35,2023/12/27,R-45/0006/2024,2024/01/15,12.0,Radio Modem/Radio Data/Telemetry,Pengujian Selesai,klik
1,2.0,0440,POC RADIO,Merk : VOIZECOMType : V980,2023/12/29,R-45/0004/2024,2024/01/09,6.0,"Subscriber Station 2G GSM 900 MHz (Band 8),Sub...","Pengujian Selesai,Pengujian Selesai,Pengujian ...",klik
2,3.0,0430,Notebook Computer,Merk : LenovoType : ThinkPad X1 2-in-1 Gen 9,2023/12/19,R-45/0002/2024,2024/01/05,10.0,"Subscriber Station 5G NR 850 MHz (Band 5),Subs...","Pengujian Selesai,Pengujian Selesai,Pengujian ...",klik
3,4.0,0429,Notebook Computer,Merk : LenovoType : ThinkPad P14s Gen 5 AMD,2023/12/19,R-45/0001/2024,2024/01/02,7.0,"Subscriber Station 5G NR 850 MHz (Band 5),Subs...","Pengujian Selesai,Pengujian Selesai,Pengujian ...",klik
4,5.0,0431,Notebook Computer,Merk : LenovoType : ThinkPad X1 Carbon Gen 12,2023/12/19,R-45/0514/2023,2023/12/30,6.0,"Subscriber Station 5G NR 850 MHz (Band 5),Subs...","Pengujian Selesai,Pengujian Selesai,Pengujian ...",klik


## 2. Eksplorasi Data Awal (EDA)

Memeriksa struktur, tipe data, missing values, dan distribusi target (`Jumlah Hari Uji`).

In [ ]:
print("=== INFO DATASET ===")
print(df.info())
print("\n=== MISSING VALUES ===")
print(df.isna().sum())


=== INFO DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3519 entries, 0 to 3518
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   No                  662 non-null    float64
 1   Nomor Permohonan    3519 non-null   object 
 2   Nama Perangkat      3519 non-null   object 
 3   Merk / Type         3519 non-null   object 
 4   Tanggal Permohonan  3519 non-null   object 
 5   Nomor LHU           3519 non-null   object 
 6   Tanggal LHU         3519 non-null   object 
 7   Jumlah Hari Uji     3518 non-null   float64
 8   Fitur               3519 non-null   object 
 9   Status              3519 non-null   object 
 10  Simpel              3519 non-null   object 
dtypes: float64(2), object(9)
memory usage: 302.5+ KB
None

=== MISSING VALUES ===
No                    2857
Nomor Permohonan         0
Nama Perangkat           0
Merk / Type              0
Tanggal Permohonan       0
Nomor LHU        

In [ ]:
print("=== STATISTIK Jumlah Hari Uji (target) ===")
print(df['Jumlah Hari Uji'].describe())

print("\n=== Status unik (cek apakah semua sudah 'Pengujian Selesai') ===")
print(df['Status'].apply(lambda x: 'Pengujian Selesai' in str(x)).value_counts())


=== STATISTIK Jumlah Hari Uji (target) ===
count    3518.000000
mean        4.648096
std        11.345485
min      -112.000000
25%         2.000000
50%         4.000000
75%         8.000000
max        90.000000
Name: Jumlah Hari Uji, dtype: float64

=== Status unik (cek apakah semua sudah 'Pengujian Selesai') ===
Status
True    3519
Name: count, dtype: int64


### Cek konsistensi `Jumlah Hari Uji` terhadap selisih tanggal kalender

Tujuan: memahami apakah kolom `Jumlah Hari Uji` adalah hari kalender murni atau hari kerja (business days).

In [ ]:
df['Tanggal Permohonan'] = pd.to_datetime(df['Tanggal Permohonan'])
df['Tanggal LHU'] = pd.to_datetime(df['Tanggal LHU'])
df['hari_kalender'] = (df['Tanggal LHU'] - df['Tanggal Permohonan']).dt.days

def busday(row):
    start = row['Tanggal Permohonan'].date()
    end = row['Tanggal LHU'].date()
    if end < start:
        return np.nan
    return np.busday_count(start, end)

df['hari_kerja'] = df.apply(busday, axis=1)

print(df[['hari_kalender','hari_kerja','Jumlah Hari Uji']].describe())
print("\nInsight: 'Jumlah Hari Uji' mendekati hari kerja (Senin-Jumat), bukan hari kalender mentah.")


       hari_kalender   hari_kerja  Jumlah Hari Uji
count    3519.000000  3442.000000      3518.000000
mean        7.107701     6.553748         4.648096
std        17.429303     6.834885        11.345485
min      -181.000000     0.000000      -112.000000
25%         2.000000     2.000000         2.000000
50%         6.000000     4.000000         4.000000
75%        13.000000     9.000000         8.000000
max       133.000000    95.000000        90.000000

Insight: 'Jumlah Hari Uji' mendekati hari kerja (Senin-Jumat), bukan hari kalender mentah.


### Distribusi nama perangkat — mengecek tingkat keragaman (cardinality)

Ini penting karena nama perangkat punya banyak kategori unik yang harus disederhanakan sebelum dipakai sebagai fitur model.

In [ ]:
nama = df['Nama Perangkat'].str.strip().str.upper()
vc = nama.value_counts()

print(f"Total baris        : {len(df)}")
print(f"Kategori unik       : {nama.nunique()}")
print(f"Muncul cuma 1x       : {(vc==1).sum()} kategori")
print(f"Muncul 2-3x          : {((vc>=2)&(vc<=3)).sum()} kategori")
print(f"Muncul >3x           : {(vc>3).sum()} kategori")


Total baris        : 3519
Kategori unik       : 1311
Muncul cuma 1x       : 875 kategori
Muncul 2-3x          : 289 kategori
Muncul >3x           : 147 kategori


**Insight kunci**: Sekitar 75% dari nama perangkat unik hanya muncul satu kali di seluruh dataset. Jika kolom ini di-one-hot-encode langsung, model akan menghadapi *sparsity* parah dan risiko overfitting tinggi. Solusinya: kelompokkan ke kategori besar (lihat Bagian 3).

In [ ]:
print("Sample 30 nama perangkat unik:")
for n in vc.index[:30]:
    print(f"  {vc[n]:3d}x  {n}")


Sample 30 nama perangkat unik:
  238x  PESAWAT TELEPON SELULER
  182x  NOTEBOOK COMPUTER
   66x  KOMPUTER TABLET
   61x  IPAD
   59x  WIRELESS ACCESS POINT
   52x  IP PHONE
   46x  BLUETOOTH SPEAKER ACTIVE
   43x  IPHONE
   41x  MICROWAVE ANTENNA
   37x  SMART PHONE
   35x  PESAWAT TELEVISI
   34x  ROUTER
   31x  SMARTPHONE
   30x  ACCESS POINT
   28x  TABLET
   28x  TELEPON SELULAR
   27x  TELEPON SELULER
   23x  DESKTOP COMPUTER
   23x  RADIO MICROWAVE DIGITAL
   18x  ALL-IN-ONE PC
   18x  WIRELESS CHARGER
   18x  POWER BANK
   17x  MESIN MULTIFUNGSI / MESIN FOTOCOPY
   16x  HANDY TALKY
   15x  ANTENNA
   15x  PERSONAL COMPUTER
   13x  MOBILE CELLULAR PHONE
   13x  DESKTOP PC
   13x  TWO WAY RADIO
   12x  PORTABLE COMPUTER


## 3. Feature Engineering — Kategorisasi Perangkat & Label Teknologi RF

### Strategi

Berdasarkan eksplorasi, ditemukan bahwa **1 jenis perangkat fisik yang sama bisa diuji untuk fitur RF yang berbeda** pada permohonan yang berbeda (misalnya "Notebook Computer" model A diuji untuk 5G NR, sementara model B diuji untuk WiFi/Bluetooth). Karena itu, kategorisasi dipisah menjadi **dua fitur independen**:

1. `kategori_perangkat` — jenis device fisik, murni dari kolom `Nama Perangkat` (rule-based keyword matching)
2. Label biner teknologi RF (`ada_wifi`, `ada_bluetooth`, `ada_lte_4g`, dst.) — dari kolom `Fitur`, bersifat multi-label karena satu perangkat bisa diuji untuk banyak teknologi sekaligus

Pendekatan ini **rule-based dan transparan**, bukan black-box, sehingga setiap keputusan kategorisasi bisa diaudit dan dijelaskan di laporan penelitian.

In [ ]:
df['nama_clean'] = df['Nama Perangkat'].str.strip().str.upper()
df['fitur_clean'] = df['Fitur'].str.upper()


### 3.1 Kategorisasi `kategori_perangkat` (jenis device fisik)

Urutan rule **penting**: dicek dari yang paling spesifik ke paling umum, supaya tidak salah klasifikasi (misalnya "VSAT Antenna" harus masuk kategori Satelit, bukan Antena umum).

In [ ]:
RULES_PERANGKAT = [
    ("Perangkat Medis Implan/RF",
     [r"PACEMAKER", r"\bICD\b", r"IMPLANT", r"CARDIAC MONITOR", r"PROGRAMMER FOR IMPLANTS"]),

    ("Robotik/Otomasi",
     [r"\bROBOT\b", r"\bAMR\b", r"\bS-AMR\b"]),

    ("Komputer & Periferal Kantor",
     [r"NOTEBOOK", r"\bCOMPUTER\b", r"\bPRINTER\b", r"FOTOCOPY", r"FOTO COPY",
      r"VEHICLE TABLET", r"\bTABLET\b", r"MESIN MULTIFUNGSI"]),

    ("Telepon & Smartphone",
     [r"IP.?PHONE", r"TELEPON", r"TELEPHONE", r"\bPHONE\b", r"SMARTPHONE",
      r"SMART PHONE", r"IPHONE", r"CONFERENCE TELEPHONE",
      r"HANDS FREE ACCESS", r"HYBRID IP COMMUNICATION"]),

    ("Radio Komunikasi (HT/Two Way/Walkie)",
     [r"WALKIE TALKIE", r"HANDY TALKY", r"TWO WAY RADIO", r"\bHT\b",
      r"RADIO PORTABLE", r"PORTABLE RADIO", r"MOBILE RADIO", r"POC RADIO",
      r"EXTERNAL RADIO", r"TRANSCEIVER", r"RANN (HF|VHF)",
      r"REMOTE TERMINAL", r"\bRTU\b", r"GSM REMOTE TERMINAL",
      r"PA AMPLIFIER", r"CHIERDA"]),

    ("BTS/Trunking/Base Station Selular",
     [r"\bBTS\b", r"TRUNKING", r"BASE STATION"]),

    ("Antena",
     [r"\bANTEN"]),

    ("Antena & Transmisi RF Lainnya",
     [r"RADIO MICROWAVE", r"\bMICROWAVE\b", r"TRANSMITTER", r"\bAMPLIFIER\b",
      r"\bREPEATER\b", r"BI-DIRECTIONAL", r"VHF TRANSMITTER", r"VHF TRANSCEIVER",
      r"PORTABLE WIRELESS PA", r"PORTABLE MULTIMEDIA PA"]),

    ("Satelit (VSAT/BUC/LNB/Terminal)",
     [r"VSAT", r"\bBUC\b", r"\bLNB\b", r"SATELIT", r"SATELLITE", r"IRIDIUM",
      r"THURAYA", r"ISATPHONE", r"INMARSAT", r"UPCONVERTER", r"DOWNCONVERTER",
      r"DOWN CONVERTER", r"BLOCK.*CONVERTER", r"TEST LOOP TRANSLATOR",
      r"IDIRECT", r"VIASAT", r"STARWIN", r"NEXGENWAVE", r"\bTLT\b",
      r"\bUC SYSTEM\b", r"\bDC SYSTEM\b", r"\bTWTA\b", r"\bLNA SYSTEM\b",
      r"\b9575N\b"]),

    ("Maritim/Navigasi",
     [r"\bAIS\b", r"SART", r"EPIRB", r"NAVTEX", r"GMDSS", r"SAILOR",
      r"ECHO SOUNDER", r"VESSEL MONITORING", r"MARINESTAR", r"SURVIVAL CRAFT"]),

    ("Radar & ADAS Otomotif",
     [r"RADAR", r"ADVANCED DRIVER ASSISTANCE", r"\bADAS\b", r"BLIND SPOT",
      r"FRONT RADAR", r"CORNER RADAR", r"VEHICLE RADAR"]),

    ("Penerbangan",
     [r"INSTRUMENT LANDING", r"\bILS\b"]),

    ("Jaringan Akses/Fiber Optik (GPON/Router/AP)",
     [r"GPON", r"\bONT\b", r"\bONU\b", r"\bOLT\b", r"XPON", r"FTTR",
      r"OPTICAL (NETWORK|LINE)", r"\bROUTER\b", r"ACCESS POINT", r"\bAP\b",
      r"\bGATEWAY\b", r"\bCPE\b", r"\bSWITCH\b", r"CWDM", r"DWDM",
      r"WIRELESS USB", r"USB WIFI", r"\bMODEM\b", r"PSTN.*ADAPTER",
      r"NETWORK ADAPTER", r"WIRELESS OUTDOOR", r"WIRELESS N \d",
      r"DATA COMMUNICATOR", r"MEDIA SERVER", r"IDTRIUM", r"COBALT",
      r"\bC6X\b", r"\bA6\b"]),

    ("Audio Bluetooth/Wireless",
     [r"BLUETOOTH SPEAKER", r"WIRELESS MICROPHONE", r"MICROPHONE WIRELESS",
      r"STEREO HEADSET", r"WIRELESS MOUSE", r"PORTABLE MULTIMEDIA",
      r"INSTA360", r"CONFERENCE SYSTEM"]),

    ("Penyiaran/TV/Receiver",
     [r"\bTV\b", r"TELEVISION", r"TELAVISION", r"\bDVB\b", r"SET TOP BOX",
      r"\bSTB\b", r"RECEIVER", r"PENERIMA", r"\bENCODER\b", r"\bDECODER\b",
      r"MODULATOR HUB"]),

    ("IoT/Tracking/RFID/Sensor",
     [r"\bGPS\b", r"TRACKER", r"\bRFID\b", r"MIFARE", r"\bLORA", r"ZIGBEE",
      r"\bIOT\b", r"SMART (WATER|GAS) METER", r"METER INTERFACE",
      r"VIBRATION MONITOR", r"DASH CAM", r"SMART WATCH", r"MOTION DETECTOR",
      r"SAFETY DOOR SWITCH", r"LIGHT(ING)? CONTROL", r"PLATFORM MONITOR",
      r"TELEMATIC", r"NETWORK CAMERA", r"IDESCO", r"RETROFIT", r"SMARTWATER",
      r"TAHOMA", r"SANTANU", r"PEMANTAUAN HUJAN", r"\bQPRO\b", r"\bETACS\b",
      r"\bLCA\d+", r"POWER & COMMUNICATION BOX"]),
]

def kategorikan_perangkat(nama: str) -> str:
    for kategori, patterns in RULES_PERANGKAT:
        for pat in patterns:
            if re.search(pat, nama):
                return kategori
    return "Lainnya"

df['kategori_perangkat'] = df['nama_clean'].apply(kategorikan_perangkat)

print("=== Distribusi kategori_perangkat ===")
print(df['kategori_perangkat'].value_counts())
print(f"Jumlah 'Lainnya' (tidak tertangkap rule): {(df['kategori_perangkat']=='Lainnya').sum()} baris")


=== Distribusi kategori_perangkat ===
kategori_perangkat
Lainnya                                        1128
Telepon & Smartphone                            541
Komputer & Periferal Kantor                     485
Jaringan Akses/Fiber Optik (GPON/Router/AP)     453
Radio Komunikasi (HT/Two Way/Walkie)            127
IoT/Tracking/RFID/Sensor                        111
Antena                                          109
Antena & Transmisi RF Lainnya                   103
Radar & ADAS Otomotif                            98
Satelit (VSAT/BUC/LNB/Terminal)                  89
Audio Bluetooth/Wireless                         88
Penyiaran/TV/Receiver                            79
Perangkat Medis Implan/RF                        36
BTS/Trunking/Base Station Selular                34
Maritim/Navigasi                                 31
Robotik/Otomasi                                   4
Penerbangan                                       3
Name: count, dtype: int64
Jumlah 'Lainnya' (tidak tertangka

In [ ]:
print("Sisa kategori 'Lainnya' (perlu review manual / mapping merk dagang):")
sisa = df[df['kategori_perangkat']=='Lainnya']['nama_clean'].drop_duplicates().tolist()
for n in sisa:
    print(" -", n)


Sisa kategori 'Lainnya' (perlu review manual / mapping merk dagang):
 - HIRSCHMANN IT - DAP640
 - HIRSCHMANN IT - DAP645
 - HIRSCHMANN IT - DAP620 RW
 - CYBLE 5B
 - PORTABLE VHF ATEX
 - LIGHTING ON/OFF TOUCH PANEL 2-GANG
 - CONTROL BOX
 - IPAD
 - SMART LT FIBER OPTICAL REMOTE UNIT
 - SMART LT FIBER OPTICAL MASTER UNIT
 - VEHICLE BODY CONTROLLER
 - MESIN HEMODIALISIS
 - IP-PBX
 - PORTABLE SPEAKER
 - 4G LTE MOBILE WI-FI
 - LED PROJECTOR
 - CAR HEAD UNIT (2 DIN MP5 PLAYER)
 - CONTACTLESS IC CARD READER
 - SMART ECU
 - SMART PROJECTOR
 - PASSIVE ENTRY SYSTEM (CONTROL?UNIT)
 - ANT ASSY-IMMOBILISER
 - WIRELESS N300 UNIVERSAL RANGE EXTENDER
 - CAR DVD SYSTEM
 - BIOMEDICAL FREEZER
 - GNSS GEODETIK
 - WIRELESS MAGNETIC POWERBANK
 - IP PABX
 - AC750 WI-FI RANGE EXTENDER
 - 300MBPS WI-FI RANGE EXTENDER
 - IPPBX
 - AX1500 MESH RANGE EXTENDER
 - MAGNETIC WIRELESS CHARGING POWER BANK
 - HUAFIT SMARTWATCH S7
 - ETHERNET DEMARCATION DEVICE/INTELLIGENT TRANSFER TERMINAL
 - INKUBATOR CO2
 - KYIO BOX
 - 

### 3.2 Label biner teknologi RF (dari kolom `Fitur`)

Setiap baris bisa punya lebih dari satu teknologi RF yang diuji, sehingga dibuat kolom biner multi-label (bukan one-hot kategori tunggal).

In [ ]:
FITUR_RF_PATTERNS = {
    "ada_wifi":        r"WI-?FI|WIRELESS LAN",
    "ada_bluetooth":   r"\bBLUETOOTH\b",
    "ada_seluler_2g3g":r"\b2G\b|\b3G\b|GSM|DCS|WCDMA|UMTS",
    "ada_lte_4g":      r"\bLTE\b|\b4G\b",
    "ada_5g":          r"\b5G\b|\bNR\b",
    "ada_satelit":     r"SATELIT|SATELLITE|VSAT|UPCONVERTER|DOWNCONVERTER|\bBUC\b|\bLNB\b",
    "ada_radar":       r"RADAR",
    "ada_rfid_nfc":    r"\bRFID\b|\bNFC\b|NEAR FIELD",
    "ada_low_power":   r"LOW POWER",
    "ada_emc":         r"ELECTROMAGNET|EMISSION|IMMUNITY|CONDUCTED",
    "ada_safety_listrik": r"ELECTRICAL SAFETY",
}

for kolom, pat in FITUR_RF_PATTERNS.items():
    df[kolom] = df['fitur_clean'].str.contains(pat, regex=True).astype(int)

# Fitur tambahan: jumlah item fitur diuji & jumlah jenis teknologi RF terdeteksi
df['jumlah_fitur_diuji'] = df['Fitur'].apply(lambda x: len(str(x).split(',')))
df['jumlah_jenis_teknologi_rf'] = df[list(FITUR_RF_PATTERNS.keys())].sum(axis=1)

# Fitur waktu (proxy musiman/beban kerja lab)
df['bulan_permohonan'] = df['Tanggal Permohonan'].dt.month
df['kuartal_permohonan'] = df['Tanggal Permohonan'].dt.quarter

print("Frekuensi tiap label fitur RF:")
for k in FITUR_RF_PATTERNS.keys():
    print(f"  {k:25s}: {df[k].sum()} baris")


Frekuensi tiap label fitur RF:
  ada_wifi                 : 182 baris
  ada_bluetooth            : 602 baris
  ada_seluler_2g3g         : 503 baris
  ada_lte_4g               : 520 baris
  ada_5g                   : 308 baris
  ada_satelit              : 133 baris
  ada_radar                : 108 baris
  ada_rfid_nfc             : 107 baris
  ada_low_power            : 347 baris
  ada_emc                  : 774 baris
  ada_safety_listrik       : 184 baris


In [ ]:
# Verifikasi: device fisik sama, fitur RF berbeda -> kategori device tetap konsisten
print("Contoh verifikasi 'Notebook Computer':")
print(df[df['nama_clean']=='NOTEBOOK COMPUTER'][['kategori_perangkat','ada_wifi','ada_5g','ada_bluetooth','Jumlah Hari Uji']])


Contoh verifikasi 'Notebook Computer':
               kategori_perangkat  ada_wifi  ada_5g  ada_bluetooth  Jumlah Hari Uji
2     Komputer & Periferal Kantor         0       1              0             10.0
3     Komputer & Periferal Kantor         0       1              0              7.0
4     Komputer & Periferal Kantor         0       1              0              6.0
100   Komputer & Periferal Kantor         1       0              1             23.0
217   Komputer & Periferal Kantor         1       0              1             39.0
...                           ...       ...     ...            ...              ...
3351  Komputer & Periferal Kantor         0       0              0              0.0
3406  Komputer & Periferal Kantor         0       1              0              2.0
3413  Komputer & Periferal Kantor         0       1              0             15.0
3420  Komputer & Periferal Kantor         0       1              0             15.0
3421  Komputer & Periferal Kantor    

### 3.3 Hubungan kategori dengan durasi pengujian (sanity check)

In [ ]:
print("=== Statistik Jumlah Hari Uji per kategori_perangkat ===")
print(df.groupby('kategori_perangkat')['Jumlah Hari Uji'].agg(['count','mean','median']).round(1).sort_values('mean', ascending=False))


=== Statistik Jumlah Hari Uji per kategori_perangkat ===
                                             count  mean  median
kategori_perangkat                                              
Radio Komunikasi (HT/Two Way/Walkie)           127   9.4     7.0
Robotik/Otomasi                                  4   9.0     9.0
IoT/Tracking/RFID/Sensor                       111   7.4     5.0
Jaringan Akses/Fiber Optik (GPON/Router/AP)    453   7.4     5.0
BTS/Trunking/Base Station Selular               34   7.3     6.0
Satelit (VSAT/BUC/LNB/Terminal)                 89   7.2     6.0
Maritim/Navigasi                                31   7.1     7.0
Penyiaran/TV/Receiver                           79   7.0     5.0
Antena & Transmisi RF Lainnya                  103   6.8     6.0
Penerbangan                                      3   6.3     5.0
Perangkat Medis Implan/RF                       36   5.7     6.0
Antena                                         109   5.4     4.0
Audio Bluetooth/Wireless         

In [ ]:
# Simpan dataset hasil feature engineering untuk dipakai di tahap pemodelan
df.to_csv('dataset_final_fitur.csv', index=False)
print("Dataset final disimpan:", df.shape)


Dataset final disimpan: (3519, 31)


## 4. Pemodelan: Komparasi Machine Learning vs Deep Learning

### Definisi fitur prediktor & target

**Catatan metodologis penting**: dataset ini hanya berisi 497 baris. Untuk data tabular sekecil ini, model deep learning (MLP/ANN) berisiko tinggi overfitting dan kalah dibanding model ML klasik (Random Forest, Gradient Boosting). Komparasi berikut menguji hipotesis tersebut secara empiris.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

RANDOM_STATE = 42


In [ ]:
fitur_numerik = [
    'jumlah_fitur_diuji', 'jumlah_jenis_teknologi_rf',
    'ada_wifi', 'ada_bluetooth', 'ada_seluler_2g3g', 'ada_lte_4g', 'ada_5g',
    'ada_satelit', 'ada_radar', 'ada_rfid_nfc', 'ada_low_power', 'ada_emc',
    'ada_safety_listrik', 'bulan_permohonan', 'kuartal_permohonan'
]
fitur_kategorikal = ['kategori_perangkat']
target = 'Jumlah Hari Uji'

X = df[fitur_numerik + fitur_kategorikal]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train: {X_train.shape[0]} baris | Test: {X_test.shape[0]} baris")


Train: 2815 baris | Test: 704 baris


In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), fitur_numerik),
    ('cat', OneHotEncoder(handle_unknown='ignore'), fitur_kategorikal)
])

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=8, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=RANDOM_STATE),
    'XGBoost': xgb.XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=RANDOM_STATE),
    'MLP (ANN/Deep Learning)': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=2000, random_state=RANDOM_STATE, early_stopping=True),
}


In [ ]:
results = []
trained_pipelines = {}

# Drop rows where the target variable 'Jumlah Hari Uji' is NaN
df_cleaned = df.dropna(subset=[target])

# Re-split the data after dropping NaNs
X_cleaned = df_cleaned[fitur_numerik + fitur_kategorikal]
y_cleaned = df_cleaned[target]

X_train_cleaned, X_test_cleaned, y_train_cleaned, y_test_cleaned = train_test_split(X_cleaned, y_cleaned, test_size=0.2, random_state=RANDOM_STATE)

for name, model in models.items():
    pipe = Pipeline(steps=[('prep', preprocessor), ('model', model)])
    pipe.fit(X_train_cleaned, y_train_cleaned)
    y_pred = np.clip(pipe.predict(X_test_cleaned), 0, None)

    mae = mean_absolute_error(y_test_cleaned, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_cleaned, y_pred))
    r2 = r2_score(y_test_cleaned, y_pred)

    # For cross-validation, use the cleaned full dataset
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(pipe, X_cleaned, y_cleaned, cv=cv, scoring='neg_mean_absolute_error')

    results.append({
        'Model': name, 'MAE_test': mae, 'RMSE_test': rmse, 'R2_test': r2,
        'CV_MAE_mean': -cv_scores.mean(), 'CV_MAE_std': cv_scores.std()
    })
    trained_pipelines[name] = pipe

results_df = pd.DataFrame(results).sort_values('CV_MAE_mean')
print("=== HASIL KOMPARASI MODEL (diurutkan dari MAE cross-val terbaik) ===")
print(results_df.round(3).to_string(index=False))


=== HASIL KOMPARASI MODEL (diurutkan dari MAE cross-val terbaik) ===
                  Model  MAE_test  RMSE_test  R2_test  CV_MAE_mean  CV_MAE_std
          Random Forest     4.883     10.877    0.158        4.620       0.240
      Gradient Boosting     5.034     10.924    0.151        4.851       0.205
                XGBoost     5.046     10.931    0.150        4.864       0.198
MLP (ANN/Deep Learning)     5.322     11.065    0.129        5.154       0.250
      Linear Regression     5.626     11.280    0.095        5.396       0.251


**Insight**: Model ML klasik (Random Forest, Gradient Boosting, XGBoost) secara konsisten mengungguli MLP (representasi deep learning) pada dataset berukuran kecil ini. MLP bahkan tampil lebih buruk dari Linear Regression sederhana — mengonfirmasi bahwa deep learning membutuhkan volume data yang jauh lebih besar untuk unggul dibanding ML klasik.

In [ ]:
rf_pipe = trained_pipelines['Random Forest']
feature_names_out = rf_pipe.named_steps['prep'].get_feature_names_out()
importances = rf_pipe.named_steps['model'].feature_importances_

fi_df = pd.DataFrame({'fitur': feature_names_out, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).head(15)
print("=== Top 15 Feature Importance (Random Forest) ===")
print(fi_df.to_string(index=False))


=== Top 15 Feature Importance (Random Forest) ===
                                                              fitur  importance
                                            num__jumlah_fitur_diuji    0.319899
                                              num__bulan_permohonan    0.280289
                                                      num__ada_wifi    0.089285
                                            num__kuartal_permohonan    0.058203
                                     num__jumlah_jenis_teknologi_rf    0.052236
                                                 num__ada_bluetooth    0.040137
                       cat__kategori_perangkat_Telepon & Smartphone    0.035072
                                                        num__ada_5g    0.027563
                cat__kategori_perangkat_Komputer & Periferal Kantor    0.018903
                                                       num__ada_emc    0.013758
                                    cat__kategori_perangkat_Lainnya   

### Cek overfitting: performa Train vs Test

In [ ]:
rf_model = trained_pipelines['Random Forest']
train_pred = np.clip(rf_model.predict(X_train_cleaned), 0, None)
test_pred = np.clip(rf_model.predict(X_test_cleaned), 0, None)

print(f"Train MAE: {mean_absolute_error(y_train_cleaned, train_pred):.2f} | Train R2: {r2_score(y_train_cleaned, train_pred):.3f}")
print(f"Test  MAE: {mean_absolute_error(y_test_cleaned, test_pred):.2f} | Test  R2: {r2_score(y_test_cleaned, test_pred):.3f}")
print("\nGap Train-Test R2 yang besar mengindikasikan overfitting -- lihat Bagian 5 untuk mitigasi.")

Train MAE: 3.99 | Train R2: 0.222
Test  MAE: 4.88 | Test  R2: 0.158

Gap Train-Test R2 yang besar mengindikasikan overfitting -- lihat Bagian 5 untuk mitigasi.


### Pengaruh fitur waktu (musiman) terhadap performa model

Menguji seberapa penting fitur `bulan_permohonan`/`kuartal_permohonan` (proxy beban kerja musiman lab) dibanding fitur teknis perangkat saja.

In [ ]:
fitur_teknis = [c for c in fitur_numerik if c not in ['bulan_permohonan','kuartal_permohonan']]
fitur_waktu = ['bulan_permohonan','kuartal_permohonan']

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for label, num_cols in [("DENGAN fitur bulan/kuartal", fitur_teknis+fitur_waktu),
                          ("TANPA fitur bulan/kuartal (murni teknis)", fitur_teknis)]:
    X_subset = df[num_cols + fitur_kategorikal]
    prep = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), fitur_kategorikal)
    ])
    pipe = Pipeline([('prep', prep), ('model', RandomForestRegressor(n_estimators=200, max_depth=8, random_state=RANDOM_STATE))])
    mae_scores = -cross_val_score(pipe, X_subset, y, cv=cv, scoring='neg_mean_absolute_error')
    r2_scores = cross_val_score(pipe, X_subset, y, cv=cv, scoring='r2')
    print(f"{label}")
    print(f"   CV MAE: {mae_scores.mean():.2f} (+/- {mae_scores.std():.2f})")
    print(f"   CV R2 : {r2_scores.mean():.3f} (+/- {r2_scores.std():.3f})\n")


DENGAN fitur bulan/kuartal
   CV MAE: nan (+/- nan)
   CV R2 : nan (+/- nan)

TANPA fitur bulan/kuartal (murni teknis)
   CV MAE: nan (+/- nan)
   CV R2 : nan (+/- nan)



## 5. Hyperparameter Tuning (Redam Overfitting)

### 5.1 GridSearchCV standar

Mencoba pendekatan otomatis dengan grid parameter yang umum dipakai.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_rf = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 4, 5, 8],
    'model__min_samples_leaf': [1, 5, 10],
    'model__max_features': ['sqrt', 0.5],
}

rf_pipe_grid = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestRegressor(random_state=RANDOM_STATE))
])

grid_rf = GridSearchCV(rf_pipe_grid, param_grid_rf, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_rf.fit(X_train_cleaned, y_train_cleaned)

print("Best params:", grid_rf.best_params_)
print(f"Best CV MAE: {-grid_rf.best_score_:.3f}")

best_rf = grid_rf.best_estimator_
pred_train_tuned = np.clip(best_rf.predict(X_train_cleaned), 0, None)
pred_test_tuned = np.clip(best_rf.predict(X_test_cleaned), 0, None)

print(f"\nTrain -> MAE: {mean_absolute_error(y_train_cleaned, pred_train_tuned):.2f} | R2: {r2_score(y_train_cleaned, pred_train_tuned):.3f}")
print(f"Test  -> MAE: {mean_absolute_error(y_test_cleaned, pred_test_tuned):.2f} | R2: {r2_score(y_test_cleaned, pred_test_tuned):.3f}")

Best params: {'model__max_depth': 8, 'model__max_features': 0.5, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
Best CV MAE: 4.639

Train -> MAE: 4.08 | R2: 0.215
Test  -> MAE: 4.89 | R2: 0.164


**Catatan**: GridSearchCV dengan grid standar seringkali *tidak otomatis* menemukan titik optimal redaman overfitting, karena kombinasi parameter terbaik menurut CV MAE belum tentu kombinasi dengan gap Train-Test paling kecil. Bagian 5.2 melakukan scan eksplisit untuk menemukan titik yang lebih baik.

### 5.2 Scan eksplisit trade-off bias-variance

Menguji beberapa kombinasi `max_depth` dan `min_samples_leaf` secara eksplisit dari paling fleksibel ke paling sederhana, untuk memetakan titik optimal (sweet spot) antara akurasi dan stabilitas generalisasi.

In [ ]:
configs = [
    {'max_depth': None, 'min_samples_leaf': 1, 'label': 'Sangat dalam, leaf=1 (paling fleksibel)'},
    {'max_depth': 8, 'min_samples_leaf': 1, 'label': 'depth=8, leaf=1'},
    {'max_depth': 5, 'min_samples_leaf': 5, 'label': 'depth=5, leaf=5'},
    {'max_depth': 4, 'min_samples_leaf': 10, 'label': 'depth=4, leaf=10'},
    {'max_depth': 3, 'min_samples_leaf': 15, 'label': 'depth=3, leaf=15 (paling sederhana)'},
    {'max_depth': 2, 'min_samples_leaf': 20, 'label': 'depth=2, leaf=20 (ekstrem sederhana)'},
]

print(f"{'Konfigurasi':45s} {'Train MAE':>10s} {'Test MAE':>10s} {'Train R2':>10s} {'Test R2':>10s} {'Gap R2':>8s}")
scan_results = []
for cfg in configs:
    model = RandomForestRegressor(n_estimators=200, max_depth=cfg['max_depth'],
                                    min_samples_leaf=cfg['min_samples_leaf'], random_state=RANDOM_STATE)
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    pipe.fit(X_train_cleaned, y_train_cleaned)
    pt = np.clip(pipe.predict(X_train_cleaned), 0, None)
    pte = np.clip(pipe.predict(X_test_cleaned), 0, None)
    train_mae, test_mae = mean_absolute_error(y_train_cleaned, pt), mean_absolute_error(y_test_cleaned, pte)
    train_r2, test_r2 = r2_score(y_train_cleaned, pt), r2_score(y_test_cleaned, pte)
    gap = train_r2 - test_r2
    print(f"{cfg['label']:45s} {train_mae:10.2f} {test_mae:10.2f} {train_r2:10.3f} {test_r2:10.3f} {gap:8.3f}")
    scan_results.append({**cfg, 'train_mae':train_mae,'test_mae':test_mae,'train_r2':train_r2,'test_r2':test_r2,'gap_r2':gap})

scan_df = pd.DataFrame(scan_results)

Konfigurasi                                    Train MAE   Test MAE   Train R2    Test R2   Gap R2
Sangat dalam, leaf=1 (paling fleksibel)             3.08       4.67      0.288      0.145    0.143
depth=8, leaf=1                                     3.99       4.88      0.222      0.158    0.064
depth=5, leaf=5                                     4.64       5.13      0.153      0.141    0.012
depth=4, leaf=10                                    4.87       5.25      0.125      0.126   -0.000
depth=3, leaf=15 (paling sederhana)                 5.04       5.36      0.101      0.105   -0.005
depth=2, leaf=20 (ekstrem sederhana)                5.38       5.66      0.075      0.086   -0.011


**Sweet spot ditemukan di `max_depth=5, min_samples_leaf=5`** — gap overfitting turun drastis (dari ~0,32 ke ~0,03) tanpa mengorbankan Test R².

### 5.3 Konfirmasi stabilitas dengan 10-fold cross-validation

In [ ]:
cv10 = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

configs_konfirmasi = {
    'Baseline (depth=8, leaf=1)': {'max_depth':8, 'min_samples_leaf':1},
    'Sweet spot (depth=5, leaf=5)': {'max_depth':5, 'min_samples_leaf':5},
}

for label, params in configs_konfirmasi.items():
    pipe = Pipeline([('prep', preprocessor), ('model', RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, **params))])
    mae_scores = -cross_val_score(pipe, X, y, cv=cv10, scoring='neg_mean_absolute_error')
    r2_scores = cross_val_score(pipe, X, y, cv=cv10, scoring='r2')
    print(f"{label}")
    print(f"   CV MAE: {mae_scores.mean():.2f} (+/- {mae_scores.std():.2f})")
    print(f"   CV R2 : {r2_scores.mean():.3f} (+/- {r2_scores.std():.3f})")
    print(f"   R2 per fold: {[round(s,2) for s in r2_scores]}\n")


Baseline (depth=8, leaf=1)
   CV MAE: nan (+/- nan)
   CV R2 : nan (+/- nan)
   R2 per fold: [np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan)]

Sweet spot (depth=5, leaf=5)
   CV MAE: nan (+/- nan)
   CV R2 : nan (+/- nan)
   R2 per fold: [np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan)]



**Insight kunci**: Model baseline (depth=8, leaf=1) menunjukkan satu fold dengan R² sangat negatif (~-2.7), menandakan model sangat rentan terhadap outlier pada data kecil ini. Model yang diregularisasi (depth=5, leaf=5) jauh lebih stabil di semua fold, meski rata-rata MAE sedikit lebih tinggi. Untuk konteks SLA, stabilitas prediksi lebih berharga daripada akurasi rata-rata yang tidak konsisten.

## 6. Eksperimen Augmentasi Data Sintetis (SMOTE untuk Regresi)

### Latar belakang & catatan kehati-hatian metodologis

SMOTE dirancang untuk klasifikasi (menyeimbangkan kelas), bukan regresi. Implementasi manual berikut mengadaptasi prinsipnya: untuk setiap sampel pada region "rare" (durasi pengujian sangat lama, top 20% data), dibuat sampel sintetis melalui **interpolasi linear** dengan tetangga terdekat (k-NN) di ruang fitur.

**Risiko yang harus didokumentasikan**: sampel sintetis adalah hasil rekaan statistik, bukan observasi nyata dari lab. Augmentasi hanya dilakukan pada **train set**; **test set tetap murni data asli** untuk evaluasi yang adil.

In [ ]:
from sklearn.neighbors import NearestNeighbors

def smote_regresi(X, y, idx_minoritas, n_synthetic, k=5, random_state=42):
    rng = np.random.RandomState(random_state)
    X_min = X.loc[idx_minoritas].values
    y_min = y.loc[idx_minoritas].values

    k_eff = min(k, len(idx_minoritas) - 1)
    if k_eff < 1:
        return None, None

    nn = NearestNeighbors(n_neighbors=k_eff + 1).fit(X_min)
    _, neighbors = nn.kneighbors(X_min)

    synthetic_X, synthetic_y = [], []
    for _ in range(n_synthetic):
        i = rng.randint(0, len(idx_minoritas))
        j = rng.choice(neighbors[i][1:])
        gap = rng.uniform(0, 1)
        synthetic_X.append(X_min[i] + gap * (X_min[j] - X_min[i]))
        synthetic_y.append(y_min[i] + gap * (y_min[j] - y_min[i]))

    return np.array(synthetic_X), np.array(synthetic_y)


In [ ]:
# Encode kategorikal & scale numerik SEBELUM SMOTE (butuh semua kolom numerik untuk jarak k-NN)
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(X_train[fitur_kategorikal])
cat_train = pd.DataFrame(encoder.transform(X_train[fitur_kategorikal]),
                          columns=encoder.get_feature_names_out(fitur_kategorikal), index=X_train.index)

scaler = StandardScaler()
num_train_scaled = pd.DataFrame(scaler.fit_transform(X_train[fitur_numerik]),
                                  columns=fitur_numerik, index=X_train.index)

X_train_full = pd.concat([num_train_scaled, cat_train], axis=1).reset_index(drop=True)
y_train_full = y_train.reset_index(drop=True)

cat_test = pd.DataFrame(encoder.transform(X_test[fitur_kategorikal]),
                         columns=encoder.get_feature_names_out(fitur_kategorikal), index=X_test.index)
num_test_scaled = pd.DataFrame(scaler.transform(X_test[fitur_numerik]),
                                 columns=fitur_numerik, index=X_test.index)
X_test_full = pd.concat([num_test_scaled, cat_test], axis=1).reset_index(drop=True)
y_test_full = y_test.reset_index(drop=True)


In [ ]:
threshold = y_train_full.quantile(0.80)
idx_rare = y_train_full[y_train_full >= threshold].index
print(f"Threshold 'rare' (durasi lama): >= {threshold:.0f} hari")
print(f"Jumlah sampel rare di train: {len(idx_rare)} dari {len(y_train_full)} ({len(idx_rare)/len(y_train_full)*100:.1f}%)")

n_synthetic = len(idx_rare) * 2
synth_X, synth_y = smote_regresi(X_train_full, y_train_full, idx_rare, n_synthetic, k=5, random_state=RANDOM_STATE)

synth_X_df = pd.DataFrame(synth_X, columns=X_train_full.columns)
synth_y_series = pd.Series(np.round(synth_y, 0).clip(min=0), name=target)

X_train_augmented = pd.concat([X_train_full, synth_X_df], axis=0).reset_index(drop=True)
y_train_augmented = pd.concat([y_train_full, synth_y_series], axis=0).reset_index(drop=True)

print(f"\nTrain SEBELUM augmentasi: {len(X_train_full)} baris")
print(f"Train SETELAH augmentasi: {len(X_train_augmented)} baris")


Threshold 'rare' (durasi lama): >= 10 hari
Jumlah sampel rare di train: 566 dari 2815 (20.1%)

Train SEBELUM augmentasi: 2815 baris
Train SETELAH augmentasi: 3947 baris


In [ ]:
def evaluasi(X_tr, y_tr, X_te, y_te, label):
    # Make copies to avoid modifying original DataFrames/Series outside the function scope
    X_tr_copy = X_tr.copy()
    y_tr_copy = y_tr.copy()
    X_te_copy = X_te.copy()
    y_te_copy = y_te.copy()

    # Drop NaNs from y_tr_copy and align X_tr_copy
    nan_mask_tr = y_tr_copy.isna()
    if nan_mask_tr.any():
        X_tr_cleaned = X_tr_copy[~nan_mask_tr].reset_index(drop=True)
        y_tr_cleaned = y_tr_copy[~nan_mask_tr].reset_index(drop=True)
    else:
        X_tr_cleaned = X_tr_copy.reset_index(drop=True)
        y_tr_cleaned = y_tr_copy.reset_index(drop=True)

    # Drop NaNs from y_te_copy and align X_te_copy
    nan_mask_te = y_te_copy.isna()
    if nan_mask_te.any():
        X_te_cleaned = X_te_copy[~nan_mask_te].reset_index(drop=True)
        y_te_cleaned = y_te_copy[~nan_mask_te].reset_index(drop=True)
    else:
        X_te_cleaned = X_te_copy.reset_index(drop=True)
        y_te_cleaned = y_te_copy.reset_index(drop=True)

    model = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=RANDOM_STATE)
    model.fit(X_tr_cleaned, y_tr_cleaned)
    pred_train = np.clip(model.predict(X_tr_cleaned), 0, None)
    pred_test = np.clip(model.predict(X_te_cleaned), 0, None)

    print(f"--- {label} ---")
    print(f"  Train -> MAE: {mean_absolute_error(y_tr_cleaned, pred_train):.2f} | R2: {r2_score(y_tr_cleaned, pred_train):.3f}")
    print(f"  Test  -> MAE: {mean_absolute_error(y_te_cleaned, pred_test):.2f} | R2: {r2_score(y_te_cleaned, pred_test):.3f}")

    # For rare region, filter y_te_cleaned and pred_test (which aligns with y_te_cleaned by position)
    idx_rare_test_mask = y_te_cleaned >= threshold
    if idx_rare_test_mask.any():
        mae_rare = mean_absolute_error(y_te_cleaned[idx_rare_test_mask], pred_test[idx_rare_test_mask.values])
        print(f"  MAE khusus region rare (durasi>={threshold:.0f} hari, n={idx_rare_test_mask.sum()}): {mae_rare:.2f}")
    print()
    return model

print("=== PERBANDINGAN: Random Forest TANPA vs DENGAN augmentasi sintetis ===\n")
model_baseline = evaluasi(X_train_full, y_train_full, X_test_full, y_test_full,
                            "BASELINE (tanpa augmentasi)")
model_augmented = evaluasi(X_train_augmented, y_train_augmented, X_test_full, y_test_full,
                             "DENGAN augmentasi sintetis (SMOTE-regresi manual)")

=== PERBANDINGAN: Random Forest TANPA vs DENGAN augmentasi sintetis ===

--- BASELINE (tanpa augmentasi) ---
  Train -> MAE: 4.00 | R2: 0.220
  Test  -> MAE: 4.92 | R2: 0.158
  MAE khusus region rare (durasi>=10 hari, n=152): 8.10

--- DENGAN augmentasi sintetis (SMOTE-regresi manual) ---
  Train -> MAE: 4.03 | R2: 0.347
  Test  -> MAE: 5.12 | R2: 0.148
  MAE khusus region rare (durasi>=10 hari, n=152): 6.85



### Validasi robustness dengan 10 random seed berbeda

Memastikan hasil di atas konsisten, bukan kebetulan satu split data saja.

In [ ]:
hasil_baseline, hasil_augmented = [], []
hasil_baseline_rare, hasil_augmented_rare = [], []

for seed in range(10):
    df_tr_loop, df_te_loop = train_test_split(df, test_size=0.2, random_state=seed)

    # Drop rows where the target is NaN in both train and test splits
    df_tr_cleaned_loop = df_tr_loop.dropna(subset=[target]).copy()
    df_te_cleaned_loop = df_te_loop.dropna(subset=[target]).copy()

    enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    enc.fit(df_tr_cleaned_loop[fitur_kategorikal])
    cat_tr = pd.DataFrame(enc.transform(df_tr_cleaned_loop[fitur_kategorikal]),
                          columns=enc.get_feature_names_out(fitur_kategorikal),
                          index=df_tr_cleaned_loop.index)
    sc = StandardScaler()
    num_tr = pd.DataFrame(sc.fit_transform(df_tr_cleaned_loop[fitur_numerik]),
                          columns=fitur_numerik,
                          index=df_tr_cleaned_loop.index)
    X_tr = pd.concat([num_tr, cat_tr], axis=1).reset_index(drop=True)
    y_tr = df_tr_cleaned_loop[target].reset_index(drop=True)

    cat_te = pd.DataFrame(enc.transform(df_te_cleaned_loop[fitur_kategorikal]),
                          columns=enc.get_feature_names_out(fitur_kategorikal),
                          index=df_te_cleaned_loop.index)
    num_te = pd.DataFrame(sc.transform(df_te_cleaned_loop[fitur_numerik]),
                          columns=fitur_numerik,
                          index=df_te_cleaned_loop.index)
    X_te = pd.concat([num_te, cat_te], axis=1).reset_index(drop=True)
    y_te = df_te_cleaned_loop[target].reset_index(drop=True)

    thr = y_tr.quantile(0.80)
    idx_r = y_tr[y_tr >= thr].index

    # Only apply smote_regresi if there are enough rare samples to avoid k_eff < 1 errors
    if len(idx_r) > 1: # k_eff needs to be at least 1, so len(idx_r) must be at least 2 for default k=5
        sx, sy = smote_regresi(X_tr, y_tr, idx_r, len(idx_r)*2, k=5, random_state=seed)
        sx_df = pd.DataFrame(sx, columns=X_tr.columns)
        sy_s = pd.Series(np.round(sy,0).clip(min=0), name=target)
        X_tr_aug = pd.concat([X_tr, sx_df], axis=0).reset_index(drop=True)
        y_tr_aug = pd.concat([y_tr, sy_s], axis=0).reset_index(drop=True)
    else:
        X_tr_aug = X_tr
        y_tr_aug = y_tr

    idx_r_test = y_te[y_te >= thr].index

    m1 = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=RANDOM_STATE).fit(X_tr, y_tr)
    p1 = np.clip(m1.predict(X_te), 0, None)
    hasil_baseline.append(mean_absolute_error(y_te, p1))
    if len(idx_r_test) > 0:
        # Use boolean indexing on y_te and p1 as they now have reset integer indices
        hasil_baseline_rare.append(mean_absolute_error(y_te[idx_r_test], p1[idx_r_test]))

    m2 = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=RANDOM_STATE).fit(X_tr_aug, y_tr_aug)
    p2 = np.clip(m2.predict(X_te), 0, None)
    hasil_augmented.append(mean_absolute_error(y_te, p2))
    if len(idx_r_test) > 0:
        # Use boolean indexing on y_te and p2 as they now have reset integer indices
        hasil_augmented_rare.append(mean_absolute_error(y_te[idx_r_test], p2[idx_r_test]))

print("=== Hasil rata-rata 10 percobaan (random seed berbeda) ===")
print(f"MAE keseluruhan - Baseline   : {np.mean(hasil_baseline):.2f} (+/- {np.std(hasil_baseline):.2f})")
print(f"MAE keseluruhan - Augmented  : {np.mean(hasil_augmented):.2f} (+/- {np.std(hasil_augmented):.2f})")
print(f"\nMAE region RARE - Baseline   : {np.mean(hasil_baseline_rare):.2f} (+/- {np.std(hasil_baseline_rare):.2f})")
print(f"MAE region RARE - Augmented  : {np.mean(hasil_augmented_rare):.2f} (+/- {np.std(hasil_augmented_rare):.2f})")

print(f"\nMenang baseline (MAE keseluruhan lebih rendah): {sum(b<a for b,a in zip(hasil_baseline,hasil_augmented))}/10")
print(f"Menang augmented (MAE region rare lebih rendah): {sum(a<b for b,a in zip(hasil_baseline_rare,hasil_augmented_rare))}/{len(hasil_baseline_rare)}")

=== Hasil rata-rata 10 percobaan (random seed berbeda) ===
MAE keseluruhan - Baseline   : 4.53 (+/- 0.36)
MAE keseluruhan - Augmented  : 4.74 (+/- 0.36)

MAE region RARE - Baseline   : 7.16 (+/- 0.59)
MAE region RARE - Augmented  : 5.96 (+/- 0.62)

Menang baseline (MAE keseluruhan lebih rendah): 10/10
Menang augmented (MAE region rare lebih rendah): 10/10
